#Importing the libraries

In [ ]:
#Import Libraries
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

#Import the Dataset

In [ ]:
!ls Dataset1.csv
df=pd.read_csv("Dataset1.csv") #placeholder dataset

# Define only pollutant (continuous) variables
pollutants = ['PAH', '1,3-Butadiene', 'O-Xylene', 'Benzene',
              'Toluene', 'Styrene', 'p-dichloro', 'Chloroform']

# Use those as features
x = df[pollutants].values
df = df.iloc[:, 9:]
# Use AMLoutcome as label
y = df['AMLoutcome']

In [ ]:
df.shape
df.head(5)


#Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler #standardizes features by removing the mean and scaling to unit variance (Z-score normalization).

# Initialize StandardScaler
scaler = StandardScaler()

# Standardize only the features
x_standardized = scaler.fit_transform(x)

# Convert back to a DataFrame (optional)
x_standardized = pd.DataFrame(x_standardized, columns=df.columns[:-1])

# Print the standardized features
print(x_standardized)

# Keep y unchanged and store if needed
y = pd.Series(y).reset_index(drop=True) # Convert y to a pandas Series before resetting index

#KPCA Tuning

In [ ]:
from sklearn.decomposition import KernelPCA
kpca_score = KernelPCA(kernel='rbf', n_components=6)  # keep only 5 principal components #initializes Kernel PCA with default parameters (RBF kernel)kpca_score = kpca_score.fit_transform(x_standardized_cleaned) #Apply KPCA
kpca_score = kpca_score.fit_transform(x_standardized) #Apply KPCA
explained_variance = np.var(kpca_score, axis=0) #Computes variance of each PC
explained_variance_ratio = explained_variance / np.sum(explained_variance)

#Plot
sns.set(style='whitegrid')
plt.plot(np.cumsum(explained_variance_ratio)) #Determines how many components needed to retain most of the information.
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance')
display(plt.show())

#Display the PC and their variance contribution
evr = explained_variance_ratio
cvr = np.cumsum(explained_variance_ratio)

kpca_df = pd.DataFrame()
kpca_df['Cumulative Variance Ratio'] = cvr
kpca_df['Explained Variance Ratio'] = evr
display(kpca_df.head(15))

#KPCA Tunned

In [ ]:
kpca = KernelPCA(n_components=2) #Creates a Kernel PCA instance that will reduce the dataset to 2 principal components.
x_transformed = kpca.fit_transform(x_standardized)

In [ ]:
x_transformed.shape

In [ ]:
print(dir(kpca))

#Determine Variable Loadings

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA

eigenvectors = kpca.eigenvectors_

# Calculate the loadings
# The loadings are essentially the dot product between original features and eigenvectors.
loadings = np.dot(x_standardized.T, eigenvectors)

# Step 4: Organize the loadings into a dataframe for better readability
loadings_df = pd.DataFrame(loadings, index=df.columns[:-1], columns=[f"PC{i+1}" for i in range(loadings.shape[1])])

# Step 5: Display the loadings
display(loadings_df)

#SVC hyperparameter tuning

In [ ]:
from sklearn.svm import SVC #Support Vector Classifier
from sklearn.model_selection import GridSearchCV #a tool that automates the process of trying different parameter combinations to find the best performing model.

# Define the parameter grid
param_grid = [
    {'C': [ 0.1, 1, 10, 100, 1000],
     'gamma': [0.0001, 0.001, 0.01, 0.1, 1],
     'kernel': ['rbf']},
    {'C': [0.1, 1, 10, 100, 1000],
     'kernel': ['linear']},
]

grid = GridSearchCV(SVC(), param_grid, verbose=2)
grid.fit(x_transformed, y)  # Fit on the 2D KernelPCA-transformed data

# Step 3: Visualize the decision boundary on the transformed data
classifier = grid.best_estimator_

# Display the best parameters and the best score
print("Best parameters found:", grid.best_params_)
print("Best cross-validation score:", grid.best_score_)

In [ ]:
grid.best_params_

{'C': 0.1, 'gamma': 0.0001, 'kernel': 'rbf'}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Assume your KernelPCA-transformed data is x_transformed and labels are y
X_set, y_set = x_transformed, y  # Use KernelPCA-transformed features and labels

# Create a meshgrid of points
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))

# Flatten the meshgrid to feed it into the classifier's decision function
grid_points = np.array([X1.ravel(), X2.ravel()]).T

# Predict decision function values for each point in the meshgrid
Z = classifier.decision_function(grid_points)

# Reshape the decision function output back to the meshgrid shape
Z = Z.reshape(X1.shape)

# Plot the decision boundary: where decision function equals 0
plt.contourf(X1, X2, Z, alpha=0.75, cmap=ListedColormap(('lightgreen', 'orangered')))

# Plot the data points
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                c=ListedColormap(('green', 'red'))(i), label=f'Class {j}')

# Add labels and title
plt.title('KernelPCA-transformed data')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.show()


#Import ICA from sklearn

In [ ]:
from sklearn.decomposition import FastICA

# Apply ICA on KPCA-Transformed Data

In [ ]:
# Define ICA with the same number of components as KPCA
ica = FastICA(n_components=x_transformed.shape[1], random_state=42)

# Apply ICA to the KPCA-transformed data
X_ica = ica.fit_transform(x_transformed)

# Now X_ica is your new feature set for classification

#Determine ICA variable Loadings

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA

ica_components = ica.components_

# Assuming x_standardized has the original feature names as columns
# Use the DataFrame version (from Feature Scaling) instead of the ndarray
loadings_df = pd.DataFrame(ica.components_,
                           index=[f"IC{i+1}" for i in range(ica.components_.shape[0])],
                           columns=[f"PC{i+1}" for i in range(ica.components_.shape[1])])  # Columns should match the KPCA output dimensions


# Step 5: Display the loadings for each component
print(loadings_df)

In [ ]:
# Step 2: Calculate correlations between original features and independent components
correlations = np.corrcoef(x_standardized.T, X_ica.T)[:x_standardized.shape[1], x_standardized.shape[1]:]

# Step 3: Convert correlations to a DataFrame for easier interpretation
correlation_df = pd.DataFrame(correlations, index=df.columns[:-1], columns=[f"IC{i+1}" for i in range(X_ica.shape[1])])

# Step 4: Display the correlations to see which original variables load most on the ICs
display(correlation_df)

# Step 5: Optionally, find the variables with the highest correlations with each IC
top_correlations_ic1 = correlation_df["IC1"].abs().sort_values(ascending=False).head(10)
top_correlations_ic2 = correlation_df["IC2"].abs().sort_values(ascending=False).head(10)

print(f"Top variables with highest correlations to IC1:\n{top_correlations_ic1}")
print(f"Top variables with highest correlations to IC2:\n{top_correlations_ic2}")


#Update the GridSearch and Model Training

In [ ]:
grid.fit(X_ica, y)  # Fit on the ICA-transformed data

# Get the best classifier
classifier = grid.best_estimator_

# Display best parameters and score
print("Best parameters found:", grid.best_params_)
print("Best cross-validation score:", grid.best_score_)


#

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have applied ICA and obtained X_ica
plt.figure(figsize=(8, 6))
plt.scatter(X_ica[:, 0], X_ica[:, 1], c=y, cmap='coolwarm', edgecolor='k', alpha=0.7)
plt.xlabel("ICA Component 1")
plt.ylabel("ICA Component 2")
plt.title("Scatter Plot of ICA Components")
plt.colorbar(label="AML Outcome (0 = No, 1 = Yes)")
plt.show()